# Exotic pricing under calibrated Bates parameters

We price path-dependent exotic options using the calibrated Bates parameters $\mathbf{p} = [\kappa, \theta, \xi, \rho, v_0, \lambda, \mu_j, \sigma_j]^\top$. Four products are evaluated by Monte Carlo simulation:

- **European call**, payoff $(S_T - K)^+$
- **Down-and-Out call**, knocked out when $\min_t S_t \leq B$
- **Down-and-In call**, activated when $\min_t S_t \leq B$
- **Asian arithmetic call**, payoff $(\bar{S} - K)^+$

Greeks $\Delta$, $\Gamma$, and $\mathcal{V}_{\text{var}}$ ($= \partial c / \partial v_0$) are computed by CRN finite differences.

In [1]:
import json, glob, os
import numpy as np
import pandas as pd

from batespricer.market import MarketEnvironment
from batespricer.models.process import BatesProcess
from batespricer.models.mc_pricer import MonteCarloPricer
from batespricer.instruments import (
    BarrierOption, BarrierType, AsianOption,
    EuropeanOption, OptionType
)

## Load calibration results

We load the most recent `_meta.json` saved by the calibration notebook.

In [2]:
patterns = [
    'calibration_*_meta.json',
    'examples/calibration_*_meta.json',
    '../calibration_*_meta.json',
    'results/calibration_*_meta.json'
]
files = []
for p in patterns:
    files.extend(glob.glob(p))
if not files:
    raise FileNotFoundError('No calibration meta file found - run 1_calibration first.')

latest = min(files, key=os.path.getctime)
with open(latest, 'r') as f:
    data = json.load(f)

p = data.get('analytical', data.get('params', data.get('monte_carlo_results', {})))
m = data.get('market', {})

r_data = m.get('r_sample', m.get('r', 0.05))
q_data = m.get('q_sample', m.get('q', 0.0))
r_val = r_data.get('1.0000Y', 0.05) if isinstance(r_data, dict) else float(r_data)
q_val = q_data.get('1.0000Y', 0.0) if isinstance(q_data, dict) else float(q_data)

env = MarketEnvironment(
    S0=m.get('S0', 100.0),
    r=r_val, q=q_val,
    kappa=p.get('kappa', 1.0), theta=p.get('theta', 0.04),
    xi=p.get('xi', 0.1), rho=p.get('rho', -0.7),
    v0=p.get('v0', 0.04),
    lamb=p.get('lamb', 0.0), mu_j=p.get('mu_j', 0.0),
    sigma_j=p.get('sigma_j', 0.0)
)

print(f"S0={env.S0:.2f}  r={env.r:.4f}  q={env.q:.4f}")
print(f"kappa={env.kappa:.3f}  theta={env.theta:.4f}  xi={env.xi:.3f}  rho={env.rho:.3f}  v0={env.v0:.4f}")
print(f"lamb={env.lamb:.3f}  mu_j={env.mu_j:.4f}  sigma_j={env.sigma_j:.4f}")

S0=6864.98  r=0.0397  q=0.0108
kappa=1.333  theta=0.0661  xi=1.087  rho=-0.801  v0=0.0243
lamb=0.751  mu_j=-0.0931  sigma_j=0.0651


## Pricing

We set $K = 1.05 \cdot S_0$, $B = 0.8 \cdot S_0$, and $T = 1$. Greeks use a 1% spot bump for $\Delta/\Gamma$ and a 10bp bump for $v_0$.

In [3]:
pricer = MonteCarloPricer(BatesProcess(env))

S0 = env.S0
K  = S0 * 1.05
B  = S0 * 0.80

products = [
    ('European Call',  EuropeanOption(K, 1.0, OptionType.CALL)),
    ('Down-Out Call',  BarrierOption(K, 1.0, B, BarrierType.DOWN_AND_OUT, OptionType.CALL)),
    ('Down-In Call',   BarrierOption(K, 1.0, B, BarrierType.DOWN_AND_IN,  OptionType.CALL)),
    ('Asian Call',     AsianOption(K, 1.0, OptionType.CALL)),
]

print(f"S0={S0:.2f}, K={K:.2f}, B={B:.2f}\n")

rows = []
for name, option in products:
    greeks = pricer.compute_greeks(option, n_paths=100_000, n_steps=1000, seed=42)
    rows.append({
        'Product': name,
        'Price':   greeks['price'],
        'Delta':   greeks['delta'],
        'Gamma':   greeks['gamma'],
        'Vega':    greeks['vega_v0']
    })
    print(f"  {name:<16} done")

df = pd.DataFrame(rows)
print('\n' + df.set_index('Product').to_string(float_format='{:.4f}'.format))

S0=6864.98, K=7208.22, B=5491.98

  European Call    done
  Down-Out Call    done
  Down-In Call     done
  Asian Call       done

                 Price  Delta   Gamma      Vega
Product                                        
European Call 416.2243 0.6088  0.0003 2448.0261
Down-Out Call 398.3037 0.6078  0.0004 2118.8124
Down-In Call   17.9206 0.0010 -0.0000  329.2137
Asian Call    147.3474 0.4874  0.0007 2160.7779


Down-and-Out and Down-and-In prices should sum to the European call.